# Skin Tone Classification Pipeline

**Fitzpatrick17k 6-class skin tone classifier** — EfficientNetV2-S + ResNet50 with fairness analysis.

## Table of Contents
1. [Data Exploration & Cleaning](#1.-Data-Exploration-&-Cleaning)
2. [Model Training](#2.-Model-Training)
3. [Evaluation & Fairness Analysis](#3.-Evaluation-&-Fairness-Analysis)

In [ ]:
import os, subprocess, sys

# Clone repo (skip if already cloned)
if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
os.chdir("/content/NST_Class")

# Install dependencies
!pip install -q -r requirements.txt

# Download dataset CSV
os.makedirs("data/images", exist_ok=True)
if not os.path.exists("data/fitzpatrick17k.csv"):
    !wget -q -O data/fitzpatrick17k.csv "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"

sys.path.insert(0, "/content/NST_Class")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
print("Setup complete!")

In [ ]:
# Google Drive: restore previous progress (set to False on first run)
RESTORE_FROM_DRIVE = True

if RESTORE_FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p data/cleaned data/images checkpoints
    if os.path.exists("/content/drive/MyDrive/NST_cleaned_data"):
        !cp -r /content/drive/MyDrive/NST_cleaned_data/* data/cleaned/
    if os.path.exists("/content/drive/MyDrive/NST_images"):
        !cp -r /content/drive/MyDrive/NST_images/* data/images/
    if os.path.exists("/content/drive/MyDrive/NST_checkpoints"):
        !cp -r /content/drive/MyDrive/NST_checkpoints/* checkpoints/
    import glob
    print(f"Restored: {len(glob.glob('data/cleaned/*.csv'))} CSVs, "
          f"{len(glob.glob('data/images/*'))} images, "
          f"{len(glob.glob('checkpoints/*.pt'))} checkpoints")
else:
    print("Skipping Drive restore (first run).")

## 1. Data Exploration & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.prepare import (
    load_metadata, validate_fitzpatrick_labels, encode_labels,
    validate_images, filter_human_images, deduplicate_images,
    compute_class_distribution, stratified_split, download_images,
)

# Configuration
CSV_PATH = "data/fitzpatrick17k.csv"
IMAGE_DIR = "data/images"
OUTPUT_DIR = "data/cleaned"
RANDOM_SEED = 42

In [ ]:
# Load metadata and show raw distribution
df = load_metadata(CSV_PATH)
print(f"Total rows: {len(df)}")

fig, ax = plt.subplots(figsize=(8, 5))
df["fitzpatrick"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Raw Fitzpatrick Skin Type Distribution")
ax.set_xlabel("Fitzpatrick Type")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Download images (parallel, batched — re-run if Colab times out)
print("Downloading images with 20 parallel workers...")
print("(Re-run this cell if it times out — it will resume where it left off)\n")
downloaded = download_images(df, IMAGE_DIR)
import glob
total_on_disk = len(glob.glob(f"{IMAGE_DIR}/*"))
print(f"\nNewly downloaded: {downloaded}")
print(f"Total images on disk: {total_on_disk}")

In [ ]:
# Full cleaning pipeline
import wandb

# Start a dedicated W&B run for data tracking
data_run = wandb.init(
    project="skin-tone-classifier",
    name="data_cleaning",
    tags=["data", "pipeline"],
    job_type="data-preparation",
)

original_count = len(df)
wandb.log({"data/original_count": original_count})

df = validate_fitzpatrick_labels(df)
dropped_labels = original_count - len(df)
print(f"After label validation: {len(df)} ({dropped_labels} dropped)")
wandb.log({"data/after_label_validation": len(df), "data/dropped_invalid_labels": dropped_labels})

before = len(df)
df = validate_images(IMAGE_DIR, df)
dropped_images = before - len(df)
print(f"After image validation: {len(df)} ({dropped_images} dropped)")
wandb.log({"data/after_image_validation": len(df), "data/dropped_missing_corrupt": dropped_images})

before = len(df)
df = filter_human_images(IMAGE_DIR, df)
dropped_nonhuman = before - len(df)
print(f"After human filter: {len(df)} ({dropped_nonhuman} dropped)")
wandb.log({"data/after_human_filter": len(df), "data/dropped_non_human": dropped_nonhuman})

before = len(df)
df = deduplicate_images(IMAGE_DIR, df)
dropped_dupes = before - len(df)
print(f"After deduplication: {len(df)} ({dropped_dupes} dropped)")
wandb.log({"data/after_deduplication": len(df), "data/dropped_duplicates": dropped_dupes})

wandb.log({
    "data/final_count": len(df),
    "data/total_dropped": original_count - len(df),
    "data/survival_rate": len(df) / original_count,
})

# Log cleaning funnel as a table
funnel = wandb.Table(columns=["Step", "Remaining", "Dropped"])
funnel.add_data("Original", original_count, 0)
funnel.add_data("Label Validation", original_count - dropped_labels, dropped_labels)
funnel.add_data("Image Validation", len(df) + dropped_nonhuman + dropped_dupes, dropped_images)
funnel.add_data("Human Filter", len(df) + dropped_dupes, dropped_nonhuman)
funnel.add_data("Deduplication", len(df), dropped_dupes)
wandb.log({"data/cleaning_funnel": funnel})

In [ ]:
# Encode labels and visualize cleaned distribution
df = encode_labels(df)

print(f"\nCleaned distribution ({len(df)} images):")
print(df["skin_tone_label"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(10, 5))
df["fitzpatrick"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Cleaned: 6-Class Fitzpatrick Distribution")
ax.set_xlabel("Fitzpatrick Type")
ax.set_ylabel("Count")
ax.set_xticklabels(["Fitz I", "Fitz II", "Fitz III", "Fitz IV", "Fitz V", "Fitz VI"], rotation=45)
plt.tight_layout()

# Log distribution chart to W&B
wandb.log({"data/class_distribution_chart": wandb.Image(fig)})
plt.show()

In [ ]:
# Class distribution and imbalance ratio
dist = compute_class_distribution(df, "fitzpatrick")
fitz_names = {1: "Fitz I", 2: "Fitz II", 3: "Fitz III", 4: "Fitz IV", 5: "Fitz V", 6: "Fitz VI"}
for cls in sorted(dist.keys()):
    info = dist[cls]
    print(f"  {fitz_names.get(cls, cls)}: {info['count']} images ({info['percentage']:.1f}%)")

imbalance_ratio = max(d["count"] for d in dist.values()) / min(d["count"] for d in dist.values())
print(f"\nImbalance ratio: {imbalance_ratio:.2f}x")

# Log per-class counts and imbalance to W&B
dist_table = wandb.Table(columns=["Fitzpatrick Type", "Count", "Percentage"])
for cls in sorted(dist.keys()):
    info = dist[cls]
    dist_table.add_data(fitz_names.get(cls, str(cls)), info["count"], round(info["percentage"], 1))
wandb.log({
    "data/class_distribution_table": dist_table,
    "data/imbalance_ratio": imbalance_ratio,
})

In [ ]:
# Stratified split and save
train_df, val_df, test_df = stratified_split(df, "skin_tone_label", (0.7, 0.15, 0.15), seed=RANDOM_SEED)
print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(os.path.join(OUTPUT_DIR, "fitzpatrick17k_cleaned.csv"), index=False)
train_df.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test.csv"), index=False)
print(f"Saved cleaned data to {OUTPUT_DIR}/")

# Log split sizes to W&B and finish data run
wandb.log({
    "data/train_size": len(train_df),
    "data/val_size": len(val_df),
    "data/test_size": len(test_df),
})
wandb.finish()
print("Data cleaning run logged to W&B.")

In [ ]:
# Drive checkpoint: save cleaned data + images
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/NST_cleaned_data /content/drive/MyDrive/NST_images
!cp -r data/cleaned/* /content/drive/MyDrive/NST_cleaned_data/
!cp -r data/images/* /content/drive/MyDrive/NST_images/
print("Saved cleaned data + images to Google Drive.")

## 2. Model Training

Two-phase training (frozen backbone then fine-tuned) for both EfficientNetV2-S and ResNet50.

In [ ]:
import torch
import wandb
from torch.utils.data import DataLoader

from src.data.dataset import FitzpatrickDataset
from src.data.transforms import get_train_transforms, get_eval_transforms
from src.models.classifier import SkinToneClassifier
from src.training.config import TrainingConfig
from src.training.trainer import Trainer, compute_class_weights
from src.utils.logging import init_wandb

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "data/cleaned"
IMAGE_DIR = "data/images"

# Reload data splits
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
# Train both backbones
BACKBONES = ["efficientnet_v2_s", "resnet50"]
all_histories = {}
use_pin_memory = DEVICE == "cuda"

# Datasets + loaders (shared across backbones)
train_dataset = FitzpatrickDataset(train_df, IMAGE_DIR, transform=get_train_transforms(224))
val_dataset = FitzpatrickDataset(val_df, IMAGE_DIR, transform=get_eval_transforms(224))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=use_pin_memory)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=use_pin_memory)

# Class weights
labels = train_df["skin_tone_label"].tolist()
weights = compute_class_weights(labels, num_classes=6)
class_weights = torch.tensor(weights, dtype=torch.float32)

for backbone in BACKBONES:
    print(f"\n{'='*60}")
    print(f"Training {backbone}")
    print(f"{'='*60}")

    config = TrainingConfig(
        backbone=backbone, num_classes=6, pretrained=True,
        freeze_backbone=True, unfreeze_after_epochs=5, epochs=20,
        batch_size=32, learning_rate=1e-4, early_stopping_patience=5,
        use_class_weights=True, wandb_project="skin-tone-classifier",
    )

    model = SkinToneClassifier(backbone, num_classes=6, pretrained=True)
    model.freeze_backbone()

    run = init_wandb(
        project=config.wandb_project, config=vars(config),
        run_name=f"{backbone}_pipeline", tags=["pipeline", backbone],
    )

    trainer = Trainer(model, config, train_loader, val_loader, class_weights, DEVICE, run)
    history = trainer.train()
    all_histories[backbone] = history

    # Save checkpoint
    Path("checkpoints").mkdir(exist_ok=True)
    torch.save(model.state_dict(), f"checkpoints/{backbone}_final.pt")
    wandb.finish()
    print(f"Saved checkpoints/{backbone}_final.pt")

In [ ]:
# Training curves side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_keys = [("loss", "Loss"), ("accuracy", "Accuracy"), ("f1", "Macro F1")]

for ax, (key, title) in zip(axes, metrics_keys):
    for backbone, history in all_histories.items():
        epochs = range(1, len(history["train"]) + 1)
        ax.plot(epochs, [m[key] for m in history["train"]], label=f"{backbone} train")
        ax.plot(epochs, [m[key] for m in history["val"]], "--", label=f"{backbone} val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Drive checkpoint: save checkpoints
!mkdir -p /content/drive/MyDrive/NST_checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/NST_checkpoints/
print("Saved checkpoints to Google Drive.")

## 3. Evaluation & Fairness Analysis

In [ ]:
from src.evaluation.metrics import compute_all_metrics
from src.evaluation.fairness import compute_fairness_gap, compare_model_fairness
from src.evaluation.confusion import plot_confusion_matrix, plot_fairness_comparison

CLASS_NAMES = ["1", "2", "3", "4", "5", "6"]
DISPLAY_NAMES = ["Fitz I", "Fitz II", "Fitz III", "Fitz IV", "Fitz V", "Fitz VI"]

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
# Evaluate all trained models
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_dataset = FitzpatrickDataset(test_df, IMAGE_DIR, transform=get_eval_transforms(224))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
print(f"Test set: {len(test_df)} images")

Path("results").mkdir(exist_ok=True)
all_metrics = {}

# Start evaluation W&B run
eval_run = wandb.init(
    project="skin-tone-classifier",
    name="evaluation",
    tags=["evaluation", "fairness"],
    job_type="evaluation",
)

for backbone in BACKBONES:
    ckpt_path = f"checkpoints/{backbone}_final.pt"
    if not os.path.exists(ckpt_path):
        print(f"Skipping {backbone} — checkpoint not found at {ckpt_path}")
        continue

    model = SkinToneClassifier(backbone, num_classes=6, pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model = model.to(DEVICE)

    y_true, y_pred, y_proba = get_predictions(model, test_loader, DEVICE)
    metrics = compute_all_metrics(y_true, y_pred, y_proba, CLASS_NAMES)
    all_metrics[backbone] = metrics

    print(f"\n{backbone}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}  Macro F1: {metrics['macro_f1']:.4f}")
    for cls, display in zip(CLASS_NAMES, DISPLAY_NAMES):
        m = metrics["per_class"][cls]
        print(f"  {display}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

    # Plot confusion matrix
    cm_fig = plot_confusion_matrix(
        metrics["confusion_matrix"], DISPLAY_NAMES,
        title=f"{backbone} Confusion Matrix",
        save_path=f"results/cm_{backbone}.png",
    )

    # Log to W&B: overall metrics
    wandb.log({
        f"eval/{backbone}/accuracy": metrics["accuracy"],
        f"eval/{backbone}/macro_f1": metrics["macro_f1"],
        f"eval/{backbone}/roc_auc": metrics["roc_auc"],
        f"eval/{backbone}/confusion_matrix": wandb.Image(cm_fig),
    })

    # Log to W&B: per-class metrics table
    per_class_table = wandb.Table(columns=["Class", "Precision", "Recall", "F1", "Support"])
    for cls, display in zip(CLASS_NAMES, DISPLAY_NAMES):
        m = metrics["per_class"][cls]
        per_class_table.add_data(display, round(m["precision"], 4), round(m["recall"], 4),
                                  round(m["f1"], 4), m["support"])
    wandb.log({f"eval/{backbone}/per_class_metrics": per_class_table})

    # Log to W&B: confusion matrix as W&B native
    wandb.log({
        f"eval/{backbone}/cm_interactive": wandb.plot.confusion_matrix(
            y_true=y_true.tolist(), preds=y_pred.tolist(),
            class_names=DISPLAY_NAMES, title=f"{backbone}",
        )
    })

    plt.close(cm_fig)

In [ ]:
# AutoML results placeholder (update after running notebook 04)
AUTOML_RESULTS_AVAILABLE = False

if not AUTOML_RESULTS_AVAILABLE:
    print("AutoML results not available. Run notebooks/04_automl_baseline.ipynb first,")
    print("then set AUTOML_RESULTS_AVAILABLE = True and fill in automl_per_class below.")

automl_per_class = {
    "1": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
    "2": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
    "3": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
    "4": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
    "5": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
    "6": {"recall": 0.0, "precision": 0.0, "f1": 0.0},
}

In [ ]:
# Fairness gap analysis
print("="*60)
print("FAIRNESS ANALYSIS")
print("="*60)

fairness_table = wandb.Table(columns=["Model", "Fairness Gap", "Best Class", "Best Recall",
                                       "Worst Class", "Worst Recall", "Significant"])

for name, metrics in all_metrics.items():
    fg = compute_fairness_gap(metrics["per_class"])
    print(f"\n{name} Fairness Gap: {fg['gap']:.2%}")
    print(f"  Best:  {fg['best_class']} ({fg['best_value']:.2%})")
    print(f"  Worst: {fg['worst_class']} ({fg['worst_value']:.2%})")
    print(f"  Significant: {fg['significant']}")

    wandb.log({
        f"fairness/{name}/gap": fg["gap"],
        f"fairness/{name}/significant": fg["significant"],
    })

    # Log per-class recall for this model
    for cls, display in zip(CLASS_NAMES, DISPLAY_NAMES):
        wandb.log({f"fairness/{name}/recall_{display}": metrics["per_class"][cls]["recall"]})

    fairness_table.add_data(name, round(fg["gap"], 4),
                            fg["best_class"], round(fg["best_value"], 4),
                            fg["worst_class"], round(fg["worst_value"], 4),
                            fg["significant"])

wandb.log({"fairness/summary": fairness_table})

In [ ]:
# Cross-model fairness comparison
model_per_class = {name: m["per_class"] for name, m in all_metrics.items()}
if AUTOML_RESULTS_AVAILABLE:
    model_per_class["AutoML"] = automl_per_class

fairness_results = compare_model_fairness(model_per_class)
fairness_fig = plot_fairness_comparison(fairness_results, save_path="results/fairness_comparison.png")

wandb.log({"fairness/comparison_chart": wandb.Image(fairness_fig)})
plt.close(fairness_fig)

In [ ]:
# Summary comparison table
print("="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<25} {'Accuracy':<10} {'Macro F1':<10} {'Fairness Gap':<15} {'Significant?'}")
print("-"*70)

summary_table = wandb.Table(columns=["Model", "Accuracy", "Macro F1", "ROC-AUC", "Fairness Gap", "Significant"])
for name, metrics in all_metrics.items():
    fg = compute_fairness_gap(metrics["per_class"])
    print(f"{name:<25} {metrics['accuracy']:<10.4f} {metrics['macro_f1']:<10.4f} {fg['gap']:<15.2%} {fg['significant']}")
    summary_table.add_data(name, round(metrics["accuracy"], 4), round(metrics["macro_f1"], 4),
                           round(metrics["roc_auc"], 4) if metrics["roc_auc"] else None,
                           round(fg["gap"], 4), fg["significant"])

wandb.log({"eval/model_comparison": summary_table})

# Finish evaluation W&B run
wandb.finish()
print("\nEvaluation run logged to W&B.")

In [ ]:
# Drive backup: save results/plots
!mkdir -p /content/drive/MyDrive/NST_results
!cp -r results/* /content/drive/MyDrive/NST_results/
print("Saved results to Google Drive.")